# 모두몰 조회 쿼리 모음
제출자: 황세진

조건 체크:
- WHERE + AND/OR/IN/BETWEEN 최소 3종 사용 → AND(Q1), OR/BETWEEN(Q2), IN(Q4)
- IS NULL / IS NOT NULL 최소 1회 사용 → IS NULL(Q3), IS NOT NULL(Q5)

## 0. 데이터 세팅 (SQLite in-memory DB)

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE customers (
  customer_id TEXT,
  name TEXT,
  country TEXT,
  signup_date TEXT,
  grade TEXT
);
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');

CREATE TABLE orders (
  order_id TEXT,
  customer_id TEXT,
  order_date TEXT,
  status TEXT,
  amount DECIMAL
);
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")
conn.commit()
print("데이터 세팅 완료")

데이터 세팅 완료


## Q1. Gold 등급이면서 Korea 국가인 고객은?

In [2]:
pd.read_sql_query("""
SELECT customer_id, name, country, grade
FROM customers
WHERE grade = 'Gold'
  AND country = 'Korea';
""", conn)

,customer_id,name,country,grade
0,C001,김민준,Korea,Gold
1,C013,권준서,Korea,Gold


**발견한 점:** Gold 등급 4명 중 Korea 국적은 2명(C001, C013)뿐이라, Gold 등급이 특정 국가에 편중되어 있지 않다.

## Q2. 2023년 4분기(10~12월)에 Paid 또는 Shipped된 주문은?

In [3]:
pd.read_sql_query("""
SELECT order_id, customer_id, order_date, status, amount
FROM orders
WHERE order_date BETWEEN '2023-10-01' AND '2023-12-31'
  AND (status = 'Paid' OR status = 'Shipped');
""", conn)

,order_id,customer_id,order_date,status,amount
0,O0007,C002,2023-10-01,Paid,158000
1,O0009,C007,2023-10-12,Shipped,410000
2,O0010,C008,2023-10-19,Paid,99000
3,O0011,C001,2023-10-23,Paid,76000
4,O0013,C010,2023-11-02,Shipped,142000
5,O0014,C004,2023-11-08,Paid,88000
6,O0016,C012,2023-11-19,Shipped,175000
7,O0018,C013,2023-11-29,Paid,320000
8,O0019,C005,2023-12-03,Paid,47000
9,O0020,C008,2023-12-09,Shipped,215000


**발견한 점:** 4분기 전체 주문(17건) 중 12건(약 70%)이 정상 결제/배송으로 이어져, 연말 성수기 취소·반품 비율이 크지 않았다.

## Q3. 국가(country) 정보가 비어있는 고객은?

In [4]:
pd.read_sql_query("""
SELECT customer_id, name, country, grade
FROM customers
WHERE country IS NULL;
""", conn)

,customer_id,name,country,grade
0,C011,오시우,None,Bronze


**발견한 점:** 전체 15명 중 1명(C011)만 국가 결측이라 심각한 품질 문제는 아니지만, country 기준 집계 시 누락 처리가 필요하다.

## Q4. Gold/Silver 등급 고객 중 취소(Cancelled)/반품(Returned)된 주문은?

In [5]:
pd.read_sql_query("""
SELECT o.order_id, c.customer_id, c.name, c.grade, o.status, o.amount
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE c.grade IN ('Gold', 'Silver')
  AND o.status IN ('Cancelled', 'Returned');
""", conn)

,order_id,customer_id,name,grade,status,amount
0,O0003,C001,김민준,Gold,Returned,45000.0
1,O0005,C004,최지우,Gold,Cancelled,NaN
2,O0017,C002,이서연,Silver,Returned,61000.0
3,O0025,C010,한예준,Silver,Cancelled,NaN
4,O0029,C004,최지우,Gold,Returned,73000.0


**발견한 점:** 취소·반품 5건 중 C004(Gold)가 2건으로 유일하게 중복 발생. 등급이 높아도 취소·반품이 없는 건 아니다 (추정치, 검증 필요: 표본이 작음).

## Q5. 결제 금액이 확정(NULL 아님)되고 20만원 이상인 고액 주문은?

In [6]:
pd.read_sql_query("""
SELECT order_id, customer_id, order_date, status, amount
FROM orders
WHERE amount IS NOT NULL
  AND amount >= 200000;
""", conn)

,order_id,customer_id,order_date,status,amount
0,O0004,C003,2023-09-15,Paid,230000
1,O0009,C007,2023-10-12,Shipped,410000
2,O0018,C013,2023-11-29,Paid,320000
3,O0020,C008,2023-12-09,Shipped,215000
4,O0024,C007,2024-01-03,Paid,268000
5,O0027,C013,2024-01-22,Shipped,405000


**발견한 점:** 30건 중 6건(20%)만 고액 주문이며, C007·C013이 각각 2건씩 포함되어 소수 고객이 고액 주문을 견인하는 경향이 보인다 (추정치, 검증 필요: 30건 표본 기준).